# AlexNet 실습: ReLU와 드롭아웃의 직관

AlexNet을 강하게 만든 ReLU와 dropout을 작은 배열로 확인합니다.


In [ ]:
# numpy는 함수 계산, matplotlib은 그래프를 그리는 데 사용합니다.
import numpy as np
import matplotlib.pyplot as plt

# 같은 결과를 얻기 위해 난수 시드를 고정합니다.
np.random.seed(12)


In [ ]:
# tanh와 ReLU 활성화 함수, 그리고 각각의 기울기를 정의합니다.
def tanh(x):
    # tanh는 큰 입력에서 출력이 -1 또는 1 근처로 포화됩니다.
    return np.tanh(x)

def tanh_grad(x):
    # tanh의 기울기는 포화될수록 0에 가까워집니다.
    return 1 - np.tanh(x) ** 2

def relu(x):
    # ReLU는 양수 입력을 그대로 통과시킵니다.
    return np.maximum(x, 0)

def relu_grad(x):
    # ReLU의 양수 영역 기울기는 1입니다.
    return (x > 0).astype(float)

x = np.linspace(-5, 5, 400)
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(x, tanh(x), label="tanh")
axes[0].plot(x, relu(x), label="ReLU")
axes[0].legend()
axes[0].set_title("activation")
axes[1].plot(x, tanh_grad(x), label="tanh grad")
axes[1].plot(x, relu_grad(x), label="ReLU grad")
axes[1].legend()
axes[1].set_title("gradient")
plt.show()


In [ ]:
# 작은 은닉층을 만들어 tanh와 ReLU의 활성값 분포를 비교합니다.
inputs = np.random.normal(0, 1, (256, 100))
weights = np.random.normal(0, 0.2, (100, 64))
pre_activation = inputs @ weights
tanh_values = tanh(pre_activation)
relu_values = relu(pre_activation)

# ReLU는 음수를 0으로 만들지만 양수 신호는 크게 누르지 않습니다.
plt.hist(tanh_values.ravel(), bins=40, alpha=0.7, label="tanh")
plt.hist(relu_values.ravel(), bins=40, alpha=0.7, label="ReLU")
plt.title("hidden activation distribution")
plt.legend()
plt.show()


In [ ]:
# 드롭아웃은 학습 중 일부 뉴런 출력을 임의로 꺼서 과적합을 줄이는 기법입니다.
def dropout(x, drop_prob=0.5, training=True):
    # 평가 시에는 모든 뉴런을 사용하므로 입력을 그대로 반환합니다.
    if not training:
        return x
    keep_prob = 1 - drop_prob
    mask = (np.random.rand(*x.shape) < keep_prob).astype(float)
    # 살아남은 값의 스케일을 보정해 평균 크기를 유지합니다.
    return x * mask / keep_prob

sample = relu_values[:5, :10]
dropped = dropout(sample, drop_prob=0.5, training=True)
print("드롭아웃 전:\n", np.round(sample, 2))
print("드롭아웃 후:\n", np.round(dropped, 2))


## 관찰 포인트

ReLU는 양수 영역에서 기울기가 잘 살아 있고, dropout은 뉴런끼리 과하게 의존하는 상황을 줄입니다.
